# Hey Bender — Custom Wake Word Training

Trains a custom openWakeWord model for **"hey bender"** using synthetic TTS audio.

**Before running:** Runtime → Change runtime type → **T4 GPU**

All outputs are saved to Google Drive so the session can safely disconnect and resume.

In [ ]:
# ── Step 0: Mount Drive ─────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
WORK_DIR = '/content/drive/MyDrive/hey_bender_training'
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)
print('Working dir:', os.getcwd())

In [ ]:
# ── Step 1: Environment setup ────────────────────────────────────────────────
# Safe to re-run — all clones and downloads are skipped if already present.
import os, subprocess, shutil, sys

def run(cmd, label=''):
    if label: print(f'[{label}]')
    r = subprocess.run(cmd, shell=True)
    if r.returncode != 0:
        raise RuntimeError(f'Failed (exit {r.returncode}): {cmd}')

# openWakeWord
if not os.path.exists('openWakeWord'):
    run('git clone -q https://github.com/dscripka/openWakeWord', 'clone openWakeWord')

# piper-sample-generator (dscripka fork — has generate_samples.py at root)
if not os.path.exists('piper-sample-generator/generate_samples.py'):
    if not os.path.exists('piper-sample-generator'):
        run('git clone -q https://github.com/dscripka/piper-sample-generator', 'clone piper-sample-generator')
    assert os.path.exists('piper-sample-generator/generate_samples.py'), \
        'generate_samples.py missing from piper-sample-generator — check the dscripka fork'
print('generate_samples.py:', os.path.exists('piper-sample-generator/generate_samples.py'))

# Piper voice model for synthetic sample generation (~700MB)
os.makedirs('piper-sample-generator/models', exist_ok=True)
piper_model = 'piper-sample-generator/models/en_US-libritts_r-medium.pt'
if not os.path.exists(piper_model):
    run(f'wget -q --show-progress -O {piper_model} '
        '"https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt"',
        'download Piper model')
print('Piper model:', 'OK' if os.path.exists(piper_model) else 'MISSING')

# Install Python deps (pinned for Python 3.12 + torchaudio 2.x compatibility)
run('pip install -q -e ./openWakeWord', 'pip install openWakeWord')
run('pip install -q webrtcvad mutagen torchinfo', 'pip install audio utils')
run('pip install -q "torchmetrics==1.2.0" "speechbrain==0.5.14"', 'pip install torch extras')
run('pip install -q "torch-audiomentations==0.11.0" "audiomentations==0.33.0"', 'pip install audiomentations')
run('pip install -q "datasets==2.14.6" "deep-phonemizer==0.0.19"', 'pip install datasets')
run('pip install -q acoustics pronouncing librosa soundfile onnxscript onnx', 'pip install remaining deps')
print('All packages installed.')

In [ ]:
# ── Step 2: Download training data ──────────────────────────────────────────
# All downloads are skipped if files already exist on Drive.
import os, datasets as hf_datasets, scipy.io.wavfile, numpy as np
from pathlib import Path
from tqdm.auto import tqdm

# MIT room impulse responses (reverb augmentation)
if not os.path.exists('mit_rirs'):
    os.makedirs('mit_rirs')
    rir_ds = hf_datasets.load_dataset('davidscripka/MIT_environmental_impulse_responses', split='train', streaming=True)
    rir_ds = rir_ds.cast_column('audio', hf_datasets.Audio(sampling_rate=16000))
    for row in tqdm(rir_ds, desc='MIT RIRs'):
        name = row['audio']['path'].split('/')[-1].replace('.mp3', '.wav')
        scipy.io.wavfile.write(f'mit_rirs/{name}', 16000, (row['audio']['array']*32767).astype(np.int16))
    print('MIT RIRs: done')
else:
    print('MIT RIRs: already on Drive')

# AudioSet background noise
if not os.path.exists('audioset_16k'):
    import subprocess
    os.makedirs('audioset', exist_ok=True)
    os.makedirs('audioset_16k')
    subprocess.run('wget -q -O audioset/bal_train09.tar https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/bal_train09.tar', shell=True)
    subprocess.run('cd audioset && tar -xf bal_train09.tar', shell=True)
    audioset_ds = hf_datasets.Dataset.from_dict({'audio': [str(i) for i in Path('audioset/audio').glob('**/*.flac')]})
    audioset_ds = audioset_ds.cast_column('audio', hf_datasets.Audio(sampling_rate=16000))
    for row in tqdm(audioset_ds, desc='AudioSet'):
        name = row['audio']['path'].split('/')[-1].replace('.flac', '.wav')
        scipy.io.wavfile.write(f'audioset_16k/{name}', 16000, (row['audio']['array']*32767).astype(np.int16))
    print('AudioSet: done')
else:
    print('AudioSet: already on Drive')

# FMA music (1 hour)
if not os.path.exists('fma') or len(os.listdir('fma')) < 10:
    os.makedirs('fma', exist_ok=True)
    fma_ds = hf_datasets.load_dataset('rudraml/fma', name='small', split='train', streaming=True)
    fma_ds = iter(fma_ds.cast_column('audio', hf_datasets.Audio(sampling_rate=16000)))
    n_hours = 1
    for i in tqdm(range(n_hours*3600//30), desc='FMA'):
        row = next(fma_ds)
        name = row['audio']['path'].split('/')[-1].replace('.mp3', '.wav')
        scipy.io.wavfile.write(f'fma/{name}', 16000, (row['audio']['array']*32767).astype(np.int16))
    print('FMA: done')
else:
    print('FMA: already on Drive')

# ACAV precomputed features (~10GB, the slow one)
if not os.path.exists('openwakeword_features_ACAV100M_2000_hrs_16bit.npy'):
    import subprocess
    print('Downloading ACAV features (~10GB, may take 15-30 min)...')
    subprocess.run('wget -q --show-progress https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy', shell=True)
    print('ACAV: done')
else:
    print('ACAV features: already on Drive')

# Validation set features (~200MB)
if not os.path.exists('validation_set_features.npy'):
    import subprocess
    subprocess.run('wget -q --show-progress https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy', shell=True)
    print('Validation features: done')
else:
    print('Validation features: already on Drive')

print('\nAll data ready.')

In [ ]:
# ── Step 3: Configure training ───────────────────────────────────────────────
# Edit TARGET_PHRASE and N_SAMPLES if needed, then run.
import yaml, os

TARGET_PHRASE = 'hey bender'   # <-- edit here if needed
N_SAMPLES = 5000               # increase to 25000 for a second, higher-quality run
STEPS = 20000                  # increase to 50000 for second run

config = yaml.load(open('openWakeWord/examples/custom_model.yml').read(), yaml.Loader)

config['target_phrase'] = [TARGET_PHRASE]
config['model_name'] = TARGET_PHRASE.replace(' ', '_')
config['n_samples'] = N_SAMPLES
config['n_samples_val'] = 1000
config['steps'] = STEPS
config['target_accuracy'] = 0.7
config['target_recall'] = 0.5
config['piper_sample_generator_path'] = os.path.abspath('piper-sample-generator')
config['piper_model'] = os.path.abspath('piper-sample-generator/models/en_US-libritts_r-medium.pt')
config['background_paths'] = ['./audioset_16k', './fma']
config['false_positive_validation_data_path'] = 'validation_set_features.npy'
config['feature_data_files'] = {'ACAV100M_sample': 'openwakeword_features_ACAV100M_2000_hrs_16bit.npy'}

with open('my_model.yaml', 'w') as f:
    yaml.dump(config, f)

print('Config written:')
for k in ['target_phrase', 'model_name', 'n_samples', 'steps', 'piper_sample_generator_path', 'piper_model']:
    print(f'  {k}: {config[k]}')

In [ ]:
# ── Step 4: Train ────────────────────────────────────────────────────────────
# Runs all 3 steps sequentially (~40-60 min total on T4).
# If generate_clips was already done, it will skip clips that already exist.
import subprocess, sys

TRAIN = f'{sys.executable} openWakeWord/openwakeword/train.py --training_config my_model.yaml'

for label, flag in [
    ('Generate clips (~10 min)', '--generate_clips'),
    ('Augment clips (~5 min)',   '--augment_clips'),
    ('Train model (~30-40 min)', '--train_model'),
]:
    print(f'\n{"="*60}\n{label}\n{"="*60}')
    r = subprocess.run(f'{TRAIN} {flag}', shell=True)
    if r.returncode != 0:
        raise RuntimeError(f'{label} failed — check output above')

In [ ]:
# ── Step 5: Save model ───────────────────────────────────────────────────────
import glob, shutil, os

onnx_files = glob.glob('**/*.onnx', recursive=True)
print('ONNX files found:', onnx_files)

saved = []
for f in onnx_files:
    name = os.path.basename(f)
    if any(x in name for x in ['hey_bender', 'custom', 'hey_sebastian']):
        dst = os.path.join(WORK_DIR, 'hey_bender_v0.1.onnx')
        shutil.copy(f, dst)
        saved.append(dst)
        print(f'Saved: {dst}')

if not saved:
    print('WARNING: No matching .onnx found. Check the file list above and copy manually.')
else:
    print('\nDownload hey_bender_v0.1.onnx from Files panel, or scp from Drive.')
    print('Deploy: scp hey_bender_v0.1.onnx pi@192.168.68.132:/home/pi/bender/models/')
    print('Config: set oww_model_path = "models/hey_bender_v0.1.onnx" in bender_config.json')